In [1]:
%matplotlib qt
import mne

#mne.viz.set_3d_backend('pyvistaqt')
from mne.coreg import Coregistration
from mne.io import read_info


import numpy as np
#%matplotlib qt
import matplotlib
#matplotlib.use('qt5agg')  # Or any other backend you want to use

import matplotlib.pyplot as plt

import pandas as pd 
import os
from os.path import join as pathjoin
from pathlib import Path

import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from time import time

from autoreject import AutoReject

import glob

import psutil
import gc
from time import time

mne.set_log_level('INFO')

import json

from mne.channels import read_dig_polhemus_isotrak  # Función para leer archivos .pos

import re
# from mne.minimum_norm import apply_inverse, make_inverse_operator
#este codigo lo dejo comentado para acostumbrarme a su suso

import brainiak
from brainiak.isc import isc 

In [ ]:
# Variables path
layer_script = "block"

#process=""
disco="g"

subj = "sub-A2004"


#subj = sys.argv[1] ## name of participant list

# Carpeta general
datadir = Path(f"{disco}:\MOUS_204")

#carpetas generales de datos
mri_dir = datadir / f"{subj}"/"anat"
meg_dir = datadir / f"{subj}"/"meg"

# Carpeta de preprocesado
output_preproc = datadir / "output_preproc"

preproc_path = output_preproc / f"preproc_{layer_script}"
preproc_path.mkdir(parents=True, exist_ok=True)
 
# Carpeta de epocas "sucias"
epochs_path = preproc_path / f"epochs_{layer_script}"
epochs_path.mkdir(parents=True, exist_ok=True) 

# Carpeta de ICA
ICA_path = preproc_path / f"ICA_{layer_script}"
ICA_path.mkdir(parents=True, exist_ok=True)

# Épocas limpias
epochs_clean_path = preproc_path / f"epochs_clean_{layer_script}"
epochs_clean_path.mkdir(parents=True, exist_ok=True)

#epocas evoked
evoked_path = Path(preproc_path) / f"evoked_{layer_script}"
evoked_path.mkdir(parents=True, exist_ok=True)
 

# Definir la carpeta de output_source antes de usarla
output_source = Path(r"g:\MOUS_204\output_source")

source_path = output_source / f"source_{layer_script}"
source_path.mkdir(parents=True, exist_ok=True)

#raw_hsp es el raw con fiducials cargados
raw_hsp_path = source_path / f"raw_hsp"
raw_hsp_path.mkdir(parents=True, exist_ok=True)

# Carpeta de forward solution
fwd_path = source_path / f"fwd"
fwd_path.mkdir(parents=True, exist_ok=True)

# Carpeta de inverse solution
inverse_path = source_path / f"inverse"
inverse_path.mkdir(parents=True, exist_ok=True)


mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)

In [3]:
subjects_dir = Path(r"\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects")

src_to_path = subjects_dir / "fsaverage" / "bem" / "fsaverage_oct-6-src.fif"
src_to = mne.read_source_spaces(src_to_path)


    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read


In [4]:
# Morrph the data to fsaverage

subjects=["sub-A2004", "sub-A2005"]
n_subjects=len(subjects)
subjects_dir = Path(r"\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects")




# Construir la ruta al archivo
src_to_path = subjects_dir / "fsaverage" / "bem" / "fsaverage_oct-6-src.fif"
src_to = mne.read_source_spaces(src_to_path)




subjects=["sub-A2004", "sub-A2005"]
n_subjects=len(subjects)

number_subjects=np.zeros(len(subjects))

for i in range(0,n_subjects):
    subj=subjects[i]
    src_orig = mne.read_source_spaces(output_source / f"{subj}_source_{layer_script}"/f"{subj}_fwd" / f"{subj}-src.fif")
    fwd = mne.read_forward_solution(output_source / f"{subj}_source_{layer_script}" /f"{subj}_fwd" / f"{subj}_fwd.fif")  
    stc_zinnen=mne.read_source_estimate(output_source / f"{subj}_source_{layer_script}"/f"{subj}_inverse" / f"{subj}_stc_zinnen_{layer_script}-stc.h5")
 
    morph_zinnen = mne.compute_source_morph(
    src=fwd['src'],                # Espacio de fuentes del sujeto
    subject_from=subj ,  # Nombre del sujeto (como está en Freesurfer)
    subject_to="fsaverage",         # Nombre del espacio común de destino
    subjects_dir=subjects_dir,      # Ruta al SUBJECTS_DIR
    spacing=5                       # Usa una resolución de vértices similar a oct6 (~4098 vértices por hemisferio))
    )
    morph_zinnen.save(output_source / f"{subj}_source_{layer_script}"/f"{subj}_inverse" / f"{subj}_morph_zinnen_{layer_script}-morph.h5", overwrite=True)
    stc_zinnen_morphed = morph_zinnen.apply(stc_zinnen)
    stc_zinnen_morphed.save(output_source / f"{subj}_source_{layer_script}"/f"{subj}_inverse" / f"{subj}_stc_zinnen_morphed_{layer_script}-stc.h5",overwrite=True)
    del src_orig
    del fwd
    del stc_zinnen
    del morph_zinnen




    Reading a source space...


    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    [done]
    2 source spaces read
Reading forward solution from g:\MOUS_204\output_source\sub-A2004_source_block\sub-A2004_fwd\sub-A2004_fwd.fif...
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    [done]
    2 source spaces read
    Desired named matrix (kind = 3523) not available
    Read MEG forward solution (8196 sources, 273 channels, free orientations)
    So

KeyboardInterrupt: 

In [5]:
subjects=["sub-A2004", "sub-A2005"]
n_subjects=len(subjects)


subj="sub-A2004"
stc_zinnen_general=mne.read_source_estimate(inverse_path / f"{subj}_stc_zinnen_morphed_{layer_script}-stc.h5")
####stc comes as an array of (n_dipoles, n_times)

data_subj=stc_zinnen_general.data
n_dipoles=data_subj.shape[0]
n_times=data_subj.shape[1]
del stc_zinnen_general
del data_subj
del subj
array_zinnen=np.zeros((n_times,n_dipoles, n_subjects))

number_subjects=np.zeros(len(subjects))

for i in range(0,n_subjects):
    subj=subjects[i]
    stc_zinnen_morphed=mne.read_source_estimate(inverse_path / f"{subj}_stc_zinnen_morphed_{layer_script}-stc.h5")
    ####stc comes as an array of (n_dipoles, n_times)
    data_subj=stc_zinnen_morphed.data
    del stc_zinnen_morphed
    ##as everything on an array is of same dtype,  number of subject)
    subj_number = int(re.findall(r'\d+', subj)[0])  # Encuentra los dígitos y convierte a entero

    # Transponer los datos (n_times x n_dipoles -> n_dipoles x n_times)
    data_subj_swapped = data_subj.T

    array_zinnen[:,:,i]=data_subj_swapped
    
    number_subjects[i]=subj_number
    del data_subj


#number_subjects_expanded = np.broadcast_to(number_subjects, (n_times,n_dipoles, n_subjects))

#array_zinnen[:, :, :] = number_subjects_expanded




In [6]:

## brainiak need an array like (n_TRs x n_voxels x n_subjects)
#so i have to create an array

#calculus of ISC correlation


iscs_zinnen= isc(data=array_zinnen, pairwise=False, summary_statistic=None, tolerate_nans=True)


In [ ]:

iscs_statistics_zinnen=brainiak.isc.compute_summary_statistic(iscs_zinnen, summary_statistic='mean', axis=None)

iscs_bootstrap_zinnen, ci_zinnen,p_zinnen, distribution_zinnen= brainiak.isc.bootstrap_isc(iscs_zinnen, pairwise=False, summary_statistic='median', n_bootstraps=1000, ci_percentile=95, side='right', random_state=None)




In [ ]:
# el phaseshift zinnen te calcula el isc y te da la distribution zinnen
observed_zinnen, p_zinnen, distribution_zinnen= brainiak.isc.phaseshift_isc(array_zinnen, pairwise=False, summary_statistic='median', n_shifts=1000, side='right', tolerate_nans=True, random_state=None)